In [1]:
import xarray as xr
import matplotlib.pyplot as plt
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np
import cartopy.crs as ccrs

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='2:00:00'
)
cluster.scale(jobs=5)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.67:45079,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [34]:
def calc_vp(Td):
    # August-Roche-Magnus approximation
    a = 17.625
    b = 243.04
    vp = xr.ufuncs.exp(((a * Td) / (b + Td)))
    return vp


def calc_q(SP, Td):
    # m_water / (m_dry air + m_water)
    # takes pressure in Pa and dew point in C
    MOLAR_MASS_RATIO = 0.622
    VP = calc_vp(Td)
    q = ((VP * MOLAR_MASS_RATIO) / (SP - (1 - MOLAR_MASS_RATIO) * VP))
    q = q.rename('Q')
    return q


def K_to_C(t):
    return (t - 273.15)

# import data

In [22]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
years = np.arange(1981, 2026, 1)
SP_ds = xr.open_zarr(path+'SP')
SP_ds = SP_ds['SP'].sel(time=SP_ds.time.dt.year.isin(years))
SP_ds

<xarray.DataArray 'SP' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<getitem, shape=(394464, 82, 121), dtype=float32, chunksize=(33112, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Attributes: (12/14)
    long_name:                                          Surface pressure
    short_name:                                         sp
    units:                                              Pa
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  128
    ecmwf_parameter:                                    134
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric surf...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [23]:
Td_ds = xr.open_zarr(path+'sfc_hourly_dew')
Td_ds = Td_ds['VAR_2D'].sel(time=Td_ds.time.dt.year.isin(years))
Td_ds

<xarray.DataArray 'VAR_2D' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<getitem, shape=(394464, 82, 121), dtype=float32, chunksize=(28256, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

# Calculate the stuff

In [24]:
Td_ds_C = K_to_C(Td_ds)
Td_ds_C

<xarray.DataArray 'VAR_2D' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<sub, shape=(394464, 82, 121), dtype=float32, chunksize=(28256, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [35]:
Q_ds = calc_q(SP_ds, Td_ds_C).drop_attrs()
Q_ds

<xarray.DataArray 'Q' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<truediv, shape=(394464, 82, 121), dtype=float32, chunksize=(28256, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [36]:
Q_ds = Q_ds.chunk({'time': 28176, 'latitude': -1, 'longitude': -1})
Q_ds

<xarray.DataArray 'Q' (time: 394464, latitude: 82, longitude: 121)> Size: 16GB
dask.array<rechunk-merge, shape=(394464, 82, 121), dtype=float32, chunksize=(28176, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [37]:
Q_ds.to_zarr(path+'sfc_Q', mode='w')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [38]:
test = xr.open_zarr(path+'sfc_Q')
test

<xarray.Dataset> Size: 16GB
Dimensions:    (time: 394464, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 3MB 1981-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    Q          (time, latitude, longitude) float32 16GB dask.array<chunksize=(28176, 82, 121), meta=np.ndarray>

In [39]:
client.shutdown()